# 🎲 Monte Carlo Simulations for Probability Estimation

---

## Introduction

**Monte Carlo simulation** is a powerful computational technique that uses **random sampling** to estimate probabilities and solve problems that might be difficult or impossible to solve analytically.

### The Core Idea

> *"Instead of deriving a formula, run the experiment thousands of times and observe what happens."*

### How It Works
1. **Define** a random process or experiment
2. **Run** thousands (or millions) of simulated trials
3. **Count** the outcomes of interest
4. **Estimate** probability as: `P(event) ≈ successes / total_trials`

### Why It's Powerful
- Works when math is too complex
- Intuitive and visual
- Used in: Finance, Physics, AI, Game Design, Drug Testing

### What We'll Explore
| Section | Experiment | Key Insight |
|---------|-----------|-------------|
| 1 | Dice Probability | P(rolling 6) = 1/6 |
| 2 | Coin Toss | P(heads) = 1/2 |
| 3 | Monty Hall Problem | Switching wins 2/3 of the time! |
| 4 | Convergence Analysis | More trials → more accurate |
| 5 | Law of Large Numbers | Variance shrinks with scale |
| 6 | Error Analysis | Error ∝ 1/√n |

In [ ]:
# ============================================================
# SETUP: Import libraries and configure global settings
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# --- Global random seed for reproducibility ---
MASTER_SEED = 42
np.random.seed(MASTER_SEED)

# --- Simulation scales ---
TRIAL_SCALES = [1_000, 10_000, 100_000]

# --- Plot styling ---
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_palette('husl')

# --- Color palette ---
COLORS = {
    'primary':   '#2563EB',
    'secondary': '#DC2626',
    'success':   '#16A34A',
    'warning':   '#D97706',
    'purple':    '#7C3AED',
    'teal':      '#0D9488',
    'theoretical': '#DC2626',
    'simulated':   '#2563EB',
}

print('✅ Libraries loaded and settings configured.')
print(f'   NumPy version  : {np.__version__}')
print(f'   Pandas version : {pd.__version__}')
print(f'   Random seed    : {MASTER_SEED}')
print(f'   Trial scales   : {[f"{t:,}" for t in TRIAL_SCALES]}')

---
## Section 1 — Dice Probability Simulation 🎲

A fair six-sided die has equal probability for each face.

**Theoretical probabilities:**
- P(rolling a 6) = 1/6 ≈ **0.1667**
- P(rolling even: 2, 4, 6) = 3/6 = **0.5000**

We'll simulate dice rolls and see how quickly our estimates converge to these values.

In [ ]:
# ============================================================
# DICE SIMULATION FUNCTIONS
# ============================================================

def roll_dice(n_trials: int, seed: int = None) -> np.ndarray:
    """Simulate rolling a fair 6-sided die n_trials times."""
    rng = np.random.default_rng(seed)
    return rng.integers(1, 7, size=n_trials)  # inclusive [1, 6]


def estimate_dice_probs(rolls: np.ndarray) -> dict:
    """Estimate P(6) and P(even) from a sequence of rolls."""
    n = len(rolls)
    return {
        'P(roll=6)':   np.sum(rolls == 6) / n,
        'P(roll even)': np.sum(rolls % 2 == 0) / n,
    }


def dice_convergence(max_trials: int, step: int, seed: int = 0) -> pd.DataFrame:
    """
    Track how P(6) evolves as we accumulate more trials.
    Returns a DataFrame with columns [trials, p_six, p_even].
    """
    all_rolls = roll_dice(max_trials, seed=seed)
    checkpoints = np.arange(step, max_trials + 1, step)
    records = []
    for n in checkpoints:
        subset = all_rolls[:n]
        records.append({
            'trials': n,
            'p_six':  np.mean(subset == 6),
            'p_even': np.mean(subset % 2 == 0),
        })
    return pd.DataFrame(records)


# --- Run simulations at all three scales ---
dice_results = {}
for n in TRIAL_SCALES:
    rolls = roll_dice(n, seed=MASTER_SEED)
    probs = estimate_dice_probs(rolls)
    dice_results[n] = {'rolls': rolls, **probs}
    print(f'Trials={n:>7,} | P(6)={probs["P(roll=6)"]:.4f} '
          f'(theory 0.1667) | P(even)={probs["P(roll even)"]:.4f} (theory 0.5000)')

In [ ]:
# ============================================================
# PLOT 1 — Dice Outcome Histogram
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
fig.suptitle('🎲 Dice Roll Outcome Distributions', fontsize=15, fontweight='bold', y=1.02)

for ax, n in zip(axes, TRIAL_SCALES):
    rolls = dice_results[n]['rolls']
    counts = np.bincount(rolls, minlength=7)[1:]  # counts for faces 1-6
    freqs  = counts / n
    bars = ax.bar(range(1, 7), freqs, color=COLORS['primary'], alpha=0.8,
                  edgecolor='white', linewidth=1.2)
    ax.axhline(1/6, color=COLORS['theoretical'], linestyle='--',
               linewidth=2, label='Theoretical (1/6)')
    # Annotate bars
    for bar, freq in zip(bars, freqs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{freq:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(f'n = {n:,} trials')
    ax.set_xlabel('Die Face')
    ax.set_ylabel('Relative Frequency')
    ax.set_xticks(range(1, 7))
    ax.set_ylim(0, 0.25)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('plot1_dice_histogram.png', bbox_inches='tight')
plt.show()
print('Plot 1 saved.')

In [ ]:
# ============================================================
# PLOT 2 — Dice Convergence Line Plot
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🎲 Dice Probability Convergence', fontsize=15, fontweight='bold')

# Compute convergence for three seeds (to show variability)
palette = [COLORS['primary'], COLORS['purple'], COLORS['teal']]

for seed_idx, seed in enumerate([0, 1, 2]):
    df = dice_convergence(max_trials=100_000, step=500, seed=seed)
    label = f'Run {seed_idx+1}'
    axes[0].plot(df['trials'], df['p_six'],  color=palette[seed_idx],
                 alpha=0.8, linewidth=1.5, label=label)
    axes[1].plot(df['trials'], df['p_even'], color=palette[seed_idx],
                 alpha=0.8, linewidth=1.5, label=label)

# Theoretical reference lines
for ax, theory, title, ylims in zip(
        axes,
        [1/6, 0.5],
        ['P(Roll = 6)', 'P(Roll = Even)'],
        [(0.10, 0.25), (0.40, 0.60)]):
    ax.axhline(theory, color=COLORS['theoretical'], linestyle='--',
               linewidth=2.5, label=f'Theoretical = {theory:.4f}')
    ax.set_title(title)
    ax.set_xlabel('Number of Trials')
    ax.set_ylabel('Estimated Probability')
    ax.set_ylim(*ylims)
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('plot2_dice_convergence.png', bbox_inches='tight')
plt.show()
print('Plot 2 saved.')

---
## Section 2 — Coin Toss Simulation 🪙

A fair coin has equal probability of landing heads or tails.

**Theoretical probability:**
- P(Heads) = **0.5000**

We encode: `1 = Heads`, `0 = Tails`

In [ ]:
# ============================================================
# COIN TOSS SIMULATION
# ============================================================

def flip_coins(n_trials: int, seed: int = None) -> np.ndarray:
    """Simulate n_trials fair coin flips. Returns 1=Heads, 0=Tails."""
    rng = np.random.default_rng(seed)
    return rng.integers(0, 2, size=n_trials)


def coin_convergence(max_trials: int, step: int, seed: int = 0) -> pd.DataFrame:
    """Track cumulative P(heads) as trials accumulate."""
    flips = flip_coins(max_trials, seed=seed)
    checkpoints = np.arange(step, max_trials + 1, step)
    records = [{'trials': n, 'p_heads': np.mean(flips[:n])} for n in checkpoints]
    return pd.DataFrame(records)


# --- Summary across scales ---
coin_results = {}
for n in TRIAL_SCALES:
    flips = flip_coins(n, seed=MASTER_SEED)
    p_heads = np.mean(flips)
    coin_results[n] = {'flips': flips, 'p_heads': p_heads}
    error = abs(p_heads - 0.5)
    print(f'Trials={n:>7,} | P(Heads)={p_heads:.5f} | Error={error:.5f}')


# ============================================================
# PLOT 3 — Coin Convergence
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🪙 Coin Toss Probability Convergence', fontsize=15, fontweight='bold')

palette = [COLORS['primary'], COLORS['purple'], COLORS['teal']]

for seed_idx, seed in enumerate([10, 11, 12]):
    df = coin_convergence(100_000, step=200, seed=seed)
    axes[0].plot(df['trials'], df['p_heads'],
                 color=palette[seed_idx], alpha=0.8, linewidth=1.5,
                 label=f'Run {seed_idx+1}')

axes[0].axhline(0.5, color=COLORS['theoretical'], linestyle='--',
                linewidth=2.5, label='Theoretical = 0.5000')
axes[0].set_title('Full Range: 0 – 100,000 trials')
axes[0].set_xlabel('Number of Trials')
axes[0].set_ylabel('P(Heads) Estimate')
axes[0].set_ylim(0.3, 0.7)
axes[0].legend(fontsize=9)
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Zoomed view — first 5,000 trials
for seed_idx, seed in enumerate([10, 11, 12]):
    df = coin_convergence(5_000, step=50, seed=seed)
    axes[1].plot(df['trials'], df['p_heads'],
                 color=palette[seed_idx], alpha=0.85, linewidth=1.5,
                 label=f'Run {seed_idx+1}')

axes[1].axhline(0.5, color=COLORS['theoretical'], linestyle='--',
                linewidth=2.5, label='Theoretical = 0.5000')
axes[1].set_title('Zoomed: 0 – 5,000 trials (high variance region)')
axes[1].set_xlabel('Number of Trials')
axes[1].set_ylabel('P(Heads) Estimate')
axes[1].set_ylim(0.3, 0.7)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('plot3_coin_convergence.png', bbox_inches='tight')
plt.show()
print('Plot 3 saved.')

---
## Section 3 — The Monty Hall Problem 🚪

One of the most famous probability puzzles in history!

### The Setup
1. There are **3 doors**: behind one is a **car 🚗**, behind the other two are **goats 🐐**
2. You **pick a door** (say Door 1)
3. The host (Monty Hall), who **knows** what's behind each door, opens **one of the remaining doors** that has a **goat**
4. You now have a choice: **Stay** with your original pick, or **Switch** to the other unopened door

### The Counterintuitive Answer

| Strategy | Theoretical Win Probability |
|----------|----------------------------|
| **Stay** | 1/3 ≈ 0.333 |
| **Switch** | 2/3 ≈ 0.667 |

> **You should ALWAYS switch!** But why? Let's simulate to confirm, then reason through it.

In [ ]:
# ============================================================
# MONTY HALL SIMULATION
# ============================================================

def simulate_monty_hall(n_trials: int, seed: int = None) -> dict:
    """
    Vectorised Monty Hall simulation.

    Returns win rates for both 'stay' and 'switch' strategies.

    Logic:
    - Car is behind a random door [0,1,2]
    - Contestant picks a random door [0,1,2]
    - Key insight: if contestant picked the car door, switch loses;
                   if contestant picked a goat door, switch wins.
    """
    rng = np.random.default_rng(seed)
    car_door      = rng.integers(0, 3, size=n_trials)  # where the car is
    chosen_door   = rng.integers(0, 3, size=n_trials)  # contestant's initial pick

    # Stay strategy: win if initial pick equals car door
    stay_wins   = (chosen_door == car_door)

    # Switch strategy: win if initial pick does NOT equal car door
    # (because host always reveals a goat, switching always lands on the car
    #  when the initial pick was a goat)
    switch_wins = (chosen_door != car_door)

    return {
        'stay_win_rate':   np.mean(stay_wins),
        'switch_win_rate': np.mean(switch_wins),
        'stay_wins':       stay_wins,
        'switch_wins':     switch_wins,
    }


def monty_hall_convergence(max_trials: int, step: int, seed: int = 0) -> pd.DataFrame:
    """Track win rates for both strategies as trials accumulate."""
    rng = np.random.default_rng(seed)
    cars    = rng.integers(0, 3, size=max_trials)
    choices = rng.integers(0, 3, size=max_trials)
    checkpoints = np.arange(step, max_trials + 1, step)
    records = []
    for n in checkpoints:
        records.append({
            'trials':          n,
            'stay_win_rate':   np.mean(choices[:n] == cars[:n]),
            'switch_win_rate': np.mean(choices[:n] != cars[:n]),
        })
    return pd.DataFrame(records)


# --- Run at all scales ---
monty_results = {}
print(f"{'Trials':>10} | {'Stay Win%':>10} | {'Switch Win%':>12} | {'Theory Stay':>12} | {'Theory Switch':>14}")
print('-' * 65)
for n in TRIAL_SCALES:
    res = simulate_monty_hall(n, seed=MASTER_SEED)
    monty_results[n] = res
    print(f"{n:>10,} | {res['stay_win_rate']:>10.4f} | "
          f"{res['switch_win_rate']:>12.4f} | "
          f"{'0.3333':>12} | {'0.6667':>14}")

In [ ]:
# ============================================================
# PLOT 4 — Monty Hall Bar Chart + Convergence
# ============================================================

fig = plt.figure(figsize=(16, 6))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 1.8], wspace=0.35)
ax_bar  = fig.add_subplot(gs[0])
ax_conv = fig.add_subplot(gs[1])
fig.suptitle('🚪 The Monty Hall Problem', fontsize=15, fontweight='bold')

# --- Bar chart: Stay vs Switch at 100,000 trials ---
res = monty_results[100_000]
strategies  = ['Stay', 'Switch']
simulated   = [res['stay_win_rate'], res['switch_win_rate']]
theoretical = [1/3, 2/3]
x = np.arange(2)
w = 0.35

b1 = ax_bar.bar(x - w/2, simulated,   w, label='Simulated (n=100k)',
                color=[COLORS['primary'], COLORS['success']], alpha=0.85)
b2 = ax_bar.bar(x + w/2, theoretical, w, label='Theoretical',
                color=[COLORS['primary'], COLORS['success']], alpha=0.35,
                edgecolor='black', linewidth=1.5, linestyle='--')

for bar, val in zip(b1, simulated):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
for bar, val in zip(b2, theoretical):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=10, color='gray')

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(strategies, fontsize=13)
ax_bar.set_ylabel('Win Probability')
ax_bar.set_ylim(0, 0.85)
ax_bar.set_title('Win Rate: Stay vs Switch\n(n = 100,000 trials)')
ax_bar.legend(fontsize=9)

# Annotation arrow
ax_bar.annotate('Switch is\n2× better!', xy=(1, 2/3), xytext=(1.55, 0.6),
                arrowprops=dict(arrowstyle='->', color='red', lw=2),
                fontsize=11, color='red', fontweight='bold')

# --- Convergence plot ---
df_mh = monty_hall_convergence(100_000, step=500, seed=42)
ax_conv.plot(df_mh['trials'], df_mh['stay_win_rate'],
             color=COLORS['primary'], linewidth=2, label='Stay')
ax_conv.plot(df_mh['trials'], df_mh['switch_win_rate'],
             color=COLORS['success'], linewidth=2, label='Switch')
ax_conv.axhline(1/3, color=COLORS['primary'], linestyle=':', linewidth=1.8, alpha=0.7,
                label='Theory Stay (1/3)')
ax_conv.axhline(2/3, color=COLORS['success'], linestyle=':', linewidth=1.8, alpha=0.7,
                label='Theory Switch (2/3)')
ax_conv.fill_between(df_mh['trials'], 1/3 - 0.02, 1/3 + 0.02,
                     color=COLORS['primary'], alpha=0.1)
ax_conv.fill_between(df_mh['trials'], 2/3 - 0.02, 2/3 + 0.02,
                     color=COLORS['success'], alpha=0.1)
ax_conv.set_xlabel('Number of Trials')
ax_conv.set_ylabel('Estimated Win Rate')
ax_conv.set_title('Convergence of Win Rates')
ax_conv.legend(fontsize=9)
ax_conv.set_ylim(0.2, 0.8)
ax_conv.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.savefig('plot4_monty_hall.png', bbox_inches='tight')
plt.show()
print('Plot 4 saved.')

### Why Does Switching Win 2/3 of the Time?

```
Initial Pick    Car Location    Host Reveals    Switch Result
─────────────   ─────────────   ────────────    ─────────────
Door 1 (Car)    Door 1          Door 2 or 3     LOSE  ← 1/3 of cases
Door 1 (Goat)   Door 2          Door 3          WIN   ← 1/3 of cases
Door 1 (Goat)   Door 3          Door 2          WIN   ← 1/3 of cases
```

**Key insight**: When you initially pick a goat (2/3 probability), the host is *forced* to reveal the other goat, making the remaining door the car. Switching wins in **both** goat-pick scenarios!

---
## Section 4 — Multi-Experiment Convergence Analysis 📈

Now let's visualize **all three experiments together** to compare how quickly each converges to its theoretical value.

In [ ]:
# ============================================================
# PLOT 5 — Multi-Experiment Convergence
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('📈 Multi-Experiment Convergence Analysis', fontsize=15, fontweight='bold')

configs = [
    {
        'ax': axes[0], 'title': 'Dice: P(Roll = 6)',
        'theory': 1/6, 'color': COLORS['primary'],
        'seeds': [0, 1, 2],
        'conv_fn': lambda seed: dice_convergence(100_000, 500, seed)['p_six'],
        'ylim': (0.10, 0.24),
    },
    {
        'ax': axes[1], 'title': 'Coin: P(Heads)',
        'theory': 0.5, 'color': COLORS['purple'],
        'seeds': [10, 11, 12],
        'conv_fn': lambda seed: coin_convergence(100_000, 500, seed)['p_heads'],
        'ylim': (0.38, 0.62),
    },
    {
        'ax': axes[2], 'title': 'Monty Hall: P(Switch Wins)',
        'theory': 2/3, 'color': COLORS['success'],
        'seeds': [20, 21, 22],
        'conv_fn': lambda seed: monty_hall_convergence(100_000, 500, seed)['switch_win_rate'],
        'ylim': (0.55, 0.78),
    },
]

x_vals = np.arange(500, 100_001, 500)

for cfg in configs:
    ax = cfg['ax']
    for i, seed in enumerate(cfg['seeds']):
        y_vals = cfg['conv_fn'](seed)
        ax.plot(x_vals, y_vals, color=cfg['color'],
                alpha=0.5 + i * 0.15, linewidth=1.4,
                label=f'Run {i+1}')
    ax.axhline(cfg['theory'], color=COLORS['theoretical'], linestyle='--',
               linewidth=2.2, label=f'Theory = {cfg["theory"]:.4f}')
    ax.set_title(cfg['title'])
    ax.set_xlabel('Number of Trials')
    ax.set_ylabel('Estimated Probability')
    ax.set_ylim(*cfg['ylim'])
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('plot5_multi_convergence.png', bbox_inches='tight')
plt.show()
print('Plot 5 saved.')

---
## Section 5 — Law of Large Numbers 📊

The **Law of Large Numbers** (LLN) states that as the number of trials increases, the sample mean converges to the expected (true) mean.

**Formally**: For a sequence of i.i.d. random variables $X_1, X_2, ..., X_n$ with mean $\mu$:

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_i \xrightarrow{n \to \infty} \mu$$

The **variance** of the estimate decreases as: $\text{Var}(\bar{X}_n) = \frac{\sigma^2}{n}$

In [ ]:
# ============================================================
# LAW OF LARGE NUMBERS — Variance Reduction Demo
# ============================================================

def compute_lln_variance(trial_sizes: np.ndarray, n_experiments: int = 200,
                          seed: int = 0) -> pd.DataFrame:
    """
    For each trial size, run n_experiments independent simulations
    and record the variance in P(heads) across experiments.
    """
    rng = np.random.default_rng(seed)
    records = []
    for n in trial_sizes:
        # Each row = one independent experiment of n flips
        all_flips = rng.integers(0, 2, size=(n_experiments, n))
        p_heads_per_exp = all_flips.mean(axis=1)
        records.append({
            'n':        n,
            'mean':     p_heads_per_exp.mean(),
            'std':      p_heads_per_exp.std(),
            'variance': p_heads_per_exp.var(),
            'theoretical_std': np.sqrt(0.25 / n),  # σ/√n for Bernoulli(0.5)
        })
    return pd.DataFrame(records)


trial_sizes = np.logspace(1, 5, num=40, dtype=int)
trial_sizes = np.unique(trial_sizes)  # remove duplicates from rounding

lln_df = compute_lln_variance(trial_sizes, n_experiments=300, seed=MASTER_SEED)

print('Law of Large Numbers — Sample Statistics')
print(lln_df[['n', 'mean', 'std', 'theoretical_std']]
      .loc[lln_df['n'].isin([10, 100, 1000, 10000, 100000])]
      .to_string(index=False, float_format='{:.5f}'.format))

In [ ]:
# ============================================================
# PLOT 6 — Law of Large Numbers Scatter + Std Dev
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('📊 Law of Large Numbers — Variance Reduction', fontsize=15, fontweight='bold')

# Left: Scatter of std dev vs trial size
ax = axes[0]
ax.scatter(lln_df['n'], lln_df['std'], color=COLORS['primary'],
           alpha=0.8, s=50, zorder=5, label='Observed Std Dev')
ax.plot(lln_df['n'], lln_df['theoretical_std'], color=COLORS['theoretical'],
        linestyle='--', linewidth=2, label='Theoretical Std Dev = √(0.25/n)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of Trials (log scale)')
ax.set_ylabel('Std Dev of P(Heads) (log scale)')
ax.set_title('Variance Reduction (Log-Log Scale)')
ax.legend(fontsize=9)

# Right: 300 independent experiments for n=100 vs n=10,000
ax2 = axes[1]
rng = np.random.default_rng(999)
n_exp = 500
small_n  = rng.integers(0, 2, (n_exp, 100)).mean(axis=1)
large_n  = rng.integers(0, 2, (n_exp, 10_000)).mean(axis=1)

bins = np.linspace(0.3, 0.7, 50)
ax2.hist(small_n, bins=bins, alpha=0.6, color=COLORS['warning'],
         label=f'n=100  (std={small_n.std():.4f})')
ax2.hist(large_n, bins=bins, alpha=0.7, color=COLORS['primary'],
         label=f'n=10,000 (std={large_n.std():.4f})')
ax2.axvline(0.5, color='black', linestyle='-', linewidth=2, label='True P = 0.5')
ax2.set_xlabel('Estimated P(Heads)')
ax2.set_ylabel('Frequency across experiments')
ax2.set_title('Distribution of Estimates\n(500 independent experiments each)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('plot6_law_of_large_numbers.png', bbox_inches='tight')
plt.show()
print('Plot 6 saved.')

---
## Section 6 — Theoretical vs Simulated Comparison 🔬

Let's create a comprehensive comparison table and visualization across all experiments.

In [ ]:
# ============================================================
# THEORETICAL vs SIMULATED SUMMARY TABLE
# ============================================================

n_max = 100_000

# Gather all simulated values at 100k trials
dice_6    = dice_results[n_max]['P(roll=6)']
dice_even = dice_results[n_max]['P(roll even)']
coin_h    = coin_results[n_max]['p_heads']
mh_stay   = monty_results[n_max]['stay_win_rate']
mh_switch = monty_results[n_max]['switch_win_rate']

comparison = pd.DataFrame([
    {'Experiment':   'Dice: P(Roll = 6)',
     'Theoretical':  1/6,
     'Simulated':    dice_6,
     'Abs Error':    abs(dice_6    - 1/6),
     'Rel Error %':  abs(dice_6    - 1/6) / (1/6) * 100},
    {'Experiment':   'Dice: P(Roll = Even)',
     'Theoretical':  0.5,
     'Simulated':    dice_even,
     'Abs Error':    abs(dice_even - 0.5),
     'Rel Error %':  abs(dice_even - 0.5) / 0.5 * 100},
    {'Experiment':   'Coin: P(Heads)',
     'Theoretical':  0.5,
     'Simulated':    coin_h,
     'Abs Error':    abs(coin_h    - 0.5),
     'Rel Error %':  abs(coin_h    - 0.5) / 0.5 * 100},
    {'Experiment':   'Monty Hall: P(Stay Wins)',
     'Theoretical':  1/3,
     'Simulated':    mh_stay,
     'Abs Error':    abs(mh_stay   - 1/3),
     'Rel Error %':  abs(mh_stay   - 1/3) / (1/3) * 100},
    {'Experiment':   'Monty Hall: P(Switch Wins)',
     'Theoretical':  2/3,
     'Simulated':    mh_switch,
     'Abs Error':    abs(mh_switch - 2/3),
     'Rel Error %':  abs(mh_switch - 2/3) / (2/3) * 100},
])

# Pretty print
pd.set_option('display.float_format', '{:.5f}'.format)
print(f'\n=== Theoretical vs Simulated (n = {n_max:,} trials) ===')
print(comparison.to_string(index=False))
print(f'\nAverage absolute error : {comparison["Abs Error"].mean():.6f}')
print(f'Average relative error : {comparison["Rel Error %"].mean():.4f}%')

In [ ]:
# ============================================================
# PLOT 7 — Theoretical vs Simulated Bar Chart
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🔬 Theoretical vs Simulated Probabilities (n = 100,000)',
             fontsize=15, fontweight='bold')

labels = [
    'Dice\nP(=6)', 'Dice\nP(Even)', 'Coin\nP(Heads)',
    'Monty\nStay', 'Monty\nSwitch'
]
x = np.arange(len(labels))
w = 0.35

# Left: grouped bar chart
b1 = axes[0].bar(x - w/2, comparison['Theoretical'], w,
                 color=COLORS['theoretical'], alpha=0.8, label='Theoretical')
b2 = axes[0].bar(x + w/2, comparison['Simulated'],   w,
                 color=COLORS['simulated'],   alpha=0.8, label='Simulated')

for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.005,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=8)

axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, fontsize=10)
axes[0].set_ylabel('Probability')
axes[0].set_title('Probability Comparison')
axes[0].legend()
axes[0].set_ylim(0, 0.85)

# Right: absolute error bar chart
colors_err = [COLORS['primary'], COLORS['purple'], COLORS['teal'],
              COLORS['warning'], COLORS['success']]
bars = axes[1].bar(x, comparison['Abs Error'] * 1000,
                   color=colors_err, alpha=0.85)
for bar, val in zip(bars, comparison['Abs Error']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.5f}', ha='center', va='bottom', fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, fontsize=10)
axes[1].set_ylabel('|Error| × 1000')
axes[1].set_title('Absolute Error per Experiment')

plt.tight_layout()
plt.savefig('plot7_theoretical_vs_simulated.png', bbox_inches='tight')
plt.show()
print('Plot 7 saved.')

---
## Section 7 — Error Analysis 📉

How does error scale with the number of trials?

By the **Central Limit Theorem**, the standard error of a Monte Carlo estimate is:

$$\text{SE} = \frac{\sigma}{\sqrt{n}}$$

This means:
- Doubling accuracy requires **4× more trials**
- Error decreases as **O(1/√n)** — the Monte Carlo convergence rate
- On a log-log plot, this appears as a **straight line with slope -0.5**

In [ ]:
# ============================================================
# ERROR ANALYSIS — |error| vs log(trials)
# ============================================================

def compute_error_vs_trials(experiment: str, n_reps: int = 50,
                             seed: int = 0) -> pd.DataFrame:
    """
    For each trial size, run n_reps independent experiments and
    compute mean absolute error from the true theoretical value.
    """
    rng = np.random.default_rng(seed)
    trial_sizes = np.unique(np.logspace(1, 5, num=30, dtype=int))
    records = []

    for n in trial_sizes:
        if experiment == 'dice_6':
            # P(6) from dice rolls
            rolls = rng.integers(1, 7, size=(n_reps, n))
            ests  = (rolls == 6).mean(axis=1)
            truth = 1/6
        elif experiment == 'coin':
            flips = rng.integers(0, 2, size=(n_reps, n))
            ests  = flips.mean(axis=1)
            truth = 0.5
        elif experiment == 'monty':
            cars    = rng.integers(0, 3, size=(n_reps, n))
            choices = rng.integers(0, 3, size=(n_reps, n))
            ests  = (choices != cars).mean(axis=1)  # switch win rate
            truth = 2/3

        mae = np.abs(ests - truth).mean()
        records.append({'n': n, 'mae': mae,
                        'theoretical_se': np.sqrt(truth * (1 - truth) / n)})

    return pd.DataFrame(records)


err_dice  = compute_error_vs_trials('dice_6', n_reps=100, seed=100)
err_coin  = compute_error_vs_trials('coin',   n_reps=100, seed=200)
err_monty = compute_error_vs_trials('monty',  n_reps=100, seed=300)

print('Error analysis complete.')
print(f'  Dice  — MAE range: [{err_dice["mae"].min():.5f}, {err_dice["mae"].max():.4f}]')
print(f'  Coin  — MAE range: [{err_coin["mae"].min():.5f}, {err_coin["mae"].max():.4f}]')
print(f'  Monty — MAE range: [{err_monty["mae"].min():.5f}, {err_monty["mae"].max():.4f}]')

In [ ]:
# ============================================================
# PLOT 8 — Error Reduction Plot
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('📉 Error Reduction Analysis — |Error| vs log(Trials)',
             fontsize=15, fontweight='bold')

# Left: All three experiments on log-log scale
ax = axes[0]
for df, label, color in [
        (err_dice,  'Dice P(=6)',      COLORS['primary']),
        (err_coin,  'Coin P(Heads)',   COLORS['purple']),
        (err_monty, 'Monty Switch',    COLORS['success']),
]:
    ax.plot(df['n'], df['mae'], 'o-', color=color,
            linewidth=1.8, markersize=4, alpha=0.85, label=f'{label} (simulated)')
    ax.plot(df['n'], df['theoretical_se'], '--', color=color,
            linewidth=1.2, alpha=0.5, label=f'{label} (theory SE)')

# Reference line: O(1/√n)
n_ref = np.logspace(1, 5, 200)
ax.plot(n_ref, 0.5 / np.sqrt(n_ref), 'k:', linewidth=2,
        alpha=0.4, label='Reference: 0.5/√n')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of Trials (log scale)')
ax.set_ylabel('Mean Absolute Error (log scale)')
ax.set_title('Error vs Trials (Log-Log) — slope ≈ -0.5')
ax.legend(fontsize=7.5, ncol=1)

# Right: Linear scale for coin toss with confidence bands
ax2 = axes[1]
n_vals = np.logspace(1, 5, 200)
se_theory = np.sqrt(0.25 / n_vals)  # σ/√n for Bernoulli(0.5)

ax2.plot(err_coin['n'], err_coin['mae'], 'o',
         color=COLORS['purple'], markersize=5, label='Simulated MAE', zorder=5)
ax2.plot(n_vals, se_theory, color=COLORS['theoretical'], linewidth=2,
         linestyle='--', label='Theoretical SE = √(0.25/n)')
ax2.fill_between(n_vals, se_theory * 0.5, se_theory * 1.5,
                 color=COLORS['theoretical'], alpha=0.12, label='±50% band')
ax2.set_xscale('log')
ax2.set_xlabel('Number of Trials (log scale)')
ax2.set_ylabel('Mean Absolute Error')
ax2.set_title('Coin Toss Error Reduction with Theoretical SE')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('plot8_error_reduction.png', bbox_inches='tight')
plt.show()
print('Plot 8 saved.')

---
## Section 8 — Comprehensive Summary Dashboard 🎯

A final combined visualization summarizing all key results.

In [ ]:
# ============================================================
# FINAL SUMMARY DASHBOARD
# ============================================================

fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#F8F9FA')
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# Title
fig.suptitle('🎲 Monte Carlo Simulation — Complete Summary Dashboard',
             fontsize=17, fontweight='bold', y=0.98)

# ── Panel 1: Dice histogram at 100k ─────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
rolls = dice_results[100_000]['rolls']
freqs = np.bincount(rolls, minlength=7)[1:] / 100_000
ax1.bar(range(1, 7), freqs, color=COLORS['primary'], alpha=0.85, edgecolor='white')
ax1.axhline(1/6, color=COLORS['theoretical'], linestyle='--', linewidth=2)
ax1.set_title('Dice Outcome Frequencies (n=100k)')
ax1.set_xlabel('Die Face')
ax1.set_ylabel('Relative Frequency')
ax1.set_xticks(range(1, 7))

# ── Panel 2: Dice P(6) convergence ──────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
df_d = dice_convergence(100_000, 500, seed=0)
ax2.plot(df_d['trials'], df_d['p_six'], color=COLORS['primary'], linewidth=1.8)
ax2.axhline(1/6, color=COLORS['theoretical'], linestyle='--', linewidth=2, label='1/6')
ax2.set_title('Dice P(=6) Convergence')
ax2.set_xlabel('Trials')
ax2.set_ylabel('P estimate')
ax2.legend(fontsize=9)
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))

# ── Panel 3: Coin convergence ────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
df_c = coin_convergence(100_000, 500, seed=10)
ax3.plot(df_c['trials'], df_c['p_heads'], color=COLORS['purple'], linewidth=1.8)
ax3.axhline(0.5, color=COLORS['theoretical'], linestyle='--', linewidth=2, label='0.5')
ax3.set_title('Coin P(Heads) Convergence')
ax3.set_xlabel('Trials')
ax3.set_ylabel('P estimate')
ax3.legend(fontsize=9)
ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))

# ── Panel 4: Monty Hall bar chart ────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
mh = monty_results[100_000]
bars = ax4.bar(['Stay', 'Switch'],
               [mh['stay_win_rate'], mh['switch_win_rate']],
               color=[COLORS['warning'], COLORS['success']], alpha=0.85)
for bar in bars:
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', fontweight='bold')
ax4.axhline(1/3, color='orange', linestyle=':', linewidth=2)
ax4.axhline(2/3, color='green',  linestyle=':', linewidth=2)
ax4.set_title('Monty Hall Win Rates (n=100k)')
ax4.set_ylabel('Win Rate')
ax4.set_ylim(0, 0.85)

# ── Panel 5: Monty Hall convergence ─────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
df_mh = monty_hall_convergence(100_000, 500, seed=42)
ax5.plot(df_mh['trials'], df_mh['stay_win_rate'],   color=COLORS['warning'], linewidth=1.8, label='Stay')
ax5.plot(df_mh['trials'], df_mh['switch_win_rate'], color=COLORS['success'], linewidth=1.8, label='Switch')
ax5.axhline(1/3, color=COLORS['warning'], linestyle=':', linewidth=1.5)
ax5.axhline(2/3, color=COLORS['success'], linestyle=':', linewidth=1.5)
ax5.set_title('Monty Hall Convergence')
ax5.set_xlabel('Trials')
ax5.set_ylabel('Win Rate')
ax5.legend(fontsize=9)
ax5.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))

# ── Panel 6: Law of Large Numbers ───────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.scatter(lln_df['n'], lln_df['std'], color=COLORS['primary'], s=30, alpha=0.8)
ax6.plot(lln_df['n'], lln_df['theoretical_std'], color=COLORS['theoretical'],
         linestyle='--', linewidth=2)
ax6.set_xscale('log')
ax6.set_yscale('log')
ax6.set_title('Law of Large Numbers')
ax6.set_xlabel('Trials (log)')
ax6.set_ylabel('Std Dev (log)')

# ── Panel 7: Error reduction ─────────────────────────────────
ax7 = fig.add_subplot(gs[2, 0])
for df, lbl, col in [
        (err_dice,  'Dice',  COLORS['primary']),
        (err_coin,  'Coin',  COLORS['purple']),
        (err_monty, 'Monty', COLORS['success'])]:
    ax7.plot(df['n'], df['mae'], 'o-', color=col, markersize=3, linewidth=1.5, label=lbl)
n_ref = np.logspace(1, 5, 200)
ax7.plot(n_ref, 0.4/np.sqrt(n_ref), 'k:', linewidth=1.5, alpha=0.5, label='~1/√n')
ax7.set_xscale('log')
ax7.set_yscale('log')
ax7.set_title('Error Reduction (Log-Log)')
ax7.set_xlabel('Trials (log)')
ax7.set_ylabel('MAE (log)')
ax7.legend(fontsize=8)

# ── Panel 8: Theoretical vs Simulated ───────────────────────
ax8 = fig.add_subplot(gs[2, 1:])
x   = np.arange(len(comparison))
w   = 0.35
ax8.bar(x - w/2, comparison['Theoretical'], w, color=COLORS['theoretical'],
        alpha=0.8, label='Theoretical')
ax8.bar(x + w/2, comparison['Simulated'],   w, color=COLORS['simulated'],
        alpha=0.8, label='Simulated (n=100k)')
short_labels = ['Dice\nP(6)', 'Dice\nEven', 'Coin\nHeads', 'MH\nStay', 'MH\nSwitch']
ax8.set_xticks(x)
ax8.set_xticklabels(short_labels, fontsize=10)
ax8.set_ylabel('Probability')
ax8.set_title('Theoretical vs Simulated — All Experiments')
ax8.legend(fontsize=9)
ax8.set_ylim(0, 0.82)

plt.savefig('plot9_dashboard.png', bbox_inches='tight', dpi=130)
plt.show()
print('Summary dashboard saved.')

---
## Section 9 — Conclusion 🎓

### What We Learned

Through this notebook, we demonstrated the power of Monte Carlo simulation as a flexible, intuitive approach to probability estimation:

| Key Insight | Finding |
|-------------|--------|
| **Convergence** | All estimates converge reliably to theoretical values with ≥ 10,000 trials |
| **Error Rate** | Error decreases as O(1/√n) — the fundamental MC convergence rate |
| **Monty Hall** | Switching wins 2/3 of the time — simulation confirmed the counterintuitive result |
| **LLN** | Variance shrinks proportionally to 1/n as sample size grows |

### Key Takeaways

1. **Monte Carlo is universal**: If you can simulate a process, you can estimate its probability
2. **More trials = more accuracy**: But with diminishing returns (4× trials → 2× accuracy)
3. **Simulation reveals truth**: Even when math seems counterintuitive (like Monty Hall!)
4. **The Law of Large Numbers is reliable**: Given enough trials, simulation always finds the truth

### Real-World Applications

| Domain | Application |
|--------|------------|
| **Finance** | Option pricing, portfolio risk, Value at Risk (VaR) |
| **Physics** | Particle transport, quantum mechanics simulations |
| **Medicine** | Clinical trial design, epidemiology models |
| **AI/ML** | Reinforcement learning, Bayesian inference, dropout |
| **Engineering** | Reliability analysis, structural stress testing |

---

> *"The most powerful tool is the ability to run an experiment a million times and observe what happens."*

**Happy Simulating! 🎲**

In [ ]:
# ============================================================
# FINAL SUMMARY PRINTOUT
# ============================================================

print('=' * 65)
print('   MONTE CARLO SIMULATION — FINAL RESULTS SUMMARY')
print('=' * 65)

print('\n📊 Experiment Results at n = 100,000 trials:')
print(f'   Dice P(=6)         : {dice_6:.5f}  (theory: {1/6:.5f},  error: {abs(dice_6-1/6):.5f})')
print(f'   Dice P(Even)       : {dice_even:.5f}  (theory: {0.5:.5f},  error: {abs(dice_even-0.5):.5f})')
print(f'   Coin P(Heads)      : {coin_h:.5f}  (theory: {0.5:.5f},  error: {abs(coin_h-0.5):.5f})')
print(f'   Monty Hall (Stay)  : {mh_stay:.5f}  (theory: {1/3:.5f},  error: {abs(mh_stay-1/3):.5f})')
print(f'   Monty Hall (Switch): {mh_switch:.5f}  (theory: {2/3:.5f},  error: {abs(mh_switch-2/3):.5f})')

print('\n📈 Error Reduction (Coin Toss):')
for n in TRIAL_SCALES:
    p = coin_results[n]['p_heads']
    err = abs(p - 0.5)
    theoretical_se = np.sqrt(0.25 / n)
    print(f'   n={n:>7,}: error={err:.5f},  theoretical SE={theoretical_se:.5f}')

print('\n🎲 Monty Hall — The Counterintuitive Result:')
print(f'   Stay strategy win rate  : {mh_stay*100:.2f}%  (expected: 33.33%)')
print(f'   Switch strategy win rate: {mh_switch*100:.2f}%  (expected: 66.67%)')
print(f'   → ALWAYS SWITCH — you win TWICE as often!')

print('\n✅ All 8 plots generated successfully!')
plots = [
    'plot1_dice_histogram.png',
    'plot2_dice_convergence.png',
    'plot3_coin_convergence.png',
    'plot4_monty_hall.png',
    'plot5_multi_convergence.png',
    'plot6_law_of_large_numbers.png',
    'plot7_theoretical_vs_simulated.png',
    'plot8_error_reduction.png',
    'plot9_dashboard.png',
]
for p in plots:
    print(f'   - {p}')

print('\n' + '=' * 65)